# Diabetes Health Indicators Analysis

An independently completed exploratory project covering data cleaning, visual analysis, binary and multiclass classification, and risk-score regression.

This public edition preserves every analysis section and all nine original code cells, with limited corrections described in `CHANGELOG.md`. Code outputs are cleared. The original CSV files are not included; configure `DIABETES_DATA_DIR` as described in the README.

The original data was unavailable during publication checks. `smoke_test.py` executes the complete code path on explicitly synthetic fixtures; its results are software checks, not research findings or clinical performance estimates.


In [ ]:
from pathlib import Path
import os
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning libraries
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Configuration
%matplotlib inline
sns.set(style="whitegrid")
import warnings
# Keep warnings visible during validation.

## Module 1: Data Exploration

**Objective:** Load the dataset, handle missing values, and remove outliers.

### 1.1 Load Data

In [ ]:
# Keep real records outside the repository.
DATA_DIR = Path(os.environ.get('DIABETES_DATA_DIR', 'data'))
required_files = ['Diabetes.csv', 'Diabetes_test.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError('Provide ' + ', '.join(missing_files) + ' in DIABETES_DATA_DIR; see data/README.md.')
df_train = pd.read_csv(DATA_DIR / 'Diabetes.csv')
df_test = pd.read_csv(DATA_DIR / 'Diabetes_test.csv')
required_columns = {'diagnosed_diabetes', 'diabetes_stage', 'diabetes_risk_score', 'bmi'}
for split, frame in [('training', df_train), ('test', df_test)]:
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f'{split} data is missing required columns: {sorted(missing)}')
print(f'Training set shape: {df_train.shape}')
print(f'Testing set shape: {df_test.shape}')


### 1.2 Handling Missing Values
I check for missing values. As per the instructions, if observations have missing values, they are dropped. 

In [ ]:
# Check for missing values
print("Missing values in Training Set before dropping:")
print(df_train.isnull().sum().sum())

# Drop rows with missing values
df_train_clean = df_train.dropna()

print(f"\nShape after dropping missing values: {df_train_clean.shape}")
print(f"Rows removed: {len(df_train) - len(df_train_clean)}")

### 1.3 Handling Outliers
I identify numerical columns and apply the Interquartile Range (IQR) method. 

In [ ]:
# Identify numerical columns for outlier detection excluding IDs and targets 
numerical_cols = [c for c in df_train_clean.select_dtypes(include=[np.number]).columns if c not in ['patient_id', 'Patient ID']]
exclude_cols = ['patient_id', 'diagnosed_diabetes', 'diabetes_risk_score', 'diabetes_stage', 'family_history_diabetes'] # Exclude ID and targets/binary
cols_to_check = [c for c in numerical_cols if c not in exclude_cols]

# Calculate Q1, Q3, and IQR
Q1 = df_train_clean[cols_to_check].quantile(0.25)
Q3 = df_train_clean[cols_to_check].quantile(0.75)
IQR = Q3 - Q1

# Define the mask for outliers based on 2.5 * IQR
outlier_mask = ~((df_train_clean[cols_to_check] < (Q1 - 2.5 * IQR)) | 
                 (df_train_clean[cols_to_check] > (Q3 + 2.5 * IQR))).any(axis=1)

df_train_final = df_train_clean[outlier_mask]

print(f"Shape after removing outliers: {df_train_final.shape}")
print(f"Observations removed due to outliers: {len(df_train_clean) - len(df_train_final)}")

## Module 2: Data Visualization

**Objective:** Explore distributions and relationships between variables.

In [ ]:
#  Distribution of the Target Variable
plt.figure(figsize=(6, 4))
sns.countplot(x='diagnosed_diabetes', data=df_train_final)
plt.title('Distribution of Diagnosed Diabetes (0=No, 1=Yes)')
plt.show()

# Correlation Heatmap
plt.figure(figsize=(12, 10))
corr_matrix = df_train_final[numerical_cols].corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm')
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

# BMI distribution by Diabetes Stage
plt.figure(figsize=(10, 6))
sns.boxplot(x='diabetes_stage', y='bmi', data=df_train_final)
plt.title('BMI Distribution by Diabetes Stage')
plt.xticks(rotation=45)
plt.show()

In this module, I explored the distribution of variables and analyzed relationships between features to understand the underlying patterns in the dataset.
1. Distribution of the Target Variable
We visualized the distribution of the target variable, diagnosed_diabetes, to understand the class balance.
Figure 1: Distribution of Diagnosed Diabetes (See Countplot above)
Analysis: The count plot reveals that the dataset is slightly imbalanced. There are approximately 3,000 observations classified as 1 (Diagnosed with Diabetes) and approximately 1,900 observations classified as 0 (No Diabetes). While there are more positive cases than negative cases, the imbalance is not extreme enough to require immediate resampling techniques (like SMOTE) for tree-based models, though it is worth noting for evaluation metrics.
2. Relationships and Patterns (Correlation Analysis)
To identify dependencies between numerical variables, I computed the correlation matrix and visualized it using a heatmap.
Figure 2: Correlation Heatmap (See Heatmap above)
Analysis:
Strong Positive Correlations: There is a distinct positive correlation (indicated by dark red blocks) between bmi and waist_to_hip_ratio, as well as between systolic_bp and diastolic_bp. This implies multicollinearity, meaning these features carry similar information.
Diabetes Risk Factors: The diabetes_risk_score shows moderate to strong positive correlations with age, bmi, glucose (if present), and insulin_level. This aligns with medical knowledge that age and obesity are significant risk factors.
Target Relationship: The target variable diagnosed_diabetes (bottom row) shows observable positive correlations with diabetes_risk_score, age, and bmi.
3. Distribution of Variables by Category
We analyzed how Body Mass Index (BMI) differs across the various stages of diabetes labeled in the dataset.
Figure 3: BMI Distribution by Diabetes Stage (See Boxplot above)
Analysis:
No Diabetes vs. Type 2: The boxplot clearly shows that the median BMI for patients with "No Diabetes" is lower (approximately 25) compared to those with "Type 2" diabetes (median approximately 26-27).
Variance: The "Type 2" and "Gestational" categories show a wider interquartile range (IQR), indicating higher variability in weight among these patients.
Outliers: There are visible outliers (circles above the whiskers) in the "Type 2", "No Diabetes", and "Pre-Diabetes" categories, indicating some patients have exceptionally high BMI values compared to the rest of their group.
Data Anomalies: There appear to be categories labeled "Type 22" and "Type 3" with very few data points (flat lines), which likely indicate typos or data entry errors in the diabetes_stage column that may need cleaning in future iterations.
Summary of Findings
The dataset has a higher prevalence of diabetic patients than non-diabetic ones.
Physical health indicators like BMI and Blood Pressure are highly correlated with each other.
There is a clear relationship where higher BMI levels are associated with advanced diabetes stages (Type 2) compared to healthy individuals.

## Module 3: Modelling

**Objective:** Build models for Binary Classification, Multiclass Classification, and Regression using 5-fold Cross-Validation and Dimension Reduction.

In [ ]:
# Keep all target columns and identifiers out of predictors.
identifier_columns = ['patient_id', 'Patient ID']
train_data = df_train_final.drop(columns=identifier_columns, errors='ignore')
test_data = df_test.drop(columns=identifier_columns, errors='ignore').dropna()
targets = ['diagnosed_diabetes', 'diabetes_stage', 'diabetes_risk_score']
feature_cols = [c for c in train_data.columns if c not in targets]
missing_features = set(feature_cols) - set(test_data.columns)
if missing_features:
    raise ValueError(f'Test data is missing predictors: {sorted(missing_features)}')
X_train_raw = train_data[feature_cols]
X_test_raw = test_data[feature_cols]
numeric_features = [c for c in feature_cols if pd.api.types.is_numeric_dtype(X_train_raw[c])]
categorical_features = [c for c in feature_cols if c not in numeric_features]

def make_model(estimator):
    # Every CV fold fits its own encoder, scaler and PCA on that fold's training rows.
    preprocess = ColumnTransformer([
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_features),
    ])
    return Pipeline([
        ('preprocess', preprocess),
        ('pca', PCA(n_components=0.95)),
        ('model', estimator),
    ])

classification_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
regression_cv = KFold(n_splits=5, shuffle=True, random_state=42)
print(f'Predictor columns: {len(feature_cols)}; encoding/scaling/PCA are fitted within each model fit.')


### 3.1 Binary Classification
**Target:** `diagnosed_diabetes` (Yes/No)
**Models:** Logistic Regression, Random Forest, SVM

In [ ]:
# Target Preparation
y_train_binary = train_data['diagnosed_diabetes']
y_test_binary = test_data['diagnosed_diabetes']

# Models
binary_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC()
}

results_binary = {}

print("--- Binary Classification Results ---")
for name, estimator in binary_models.items():
    model = make_model(estimator)
    # 5-fold CV
    cv_scores = cross_val_score(model, X_train_raw, y_train_binary, cv=classification_cv, scoring='accuracy')
    
    # Fit on full training data
    model.fit(X_train_raw, y_train_binary)
    
    # Predict on test data
    y_pred = model.predict(X_test_raw)
    
    # Metrics
    acc = accuracy_score(y_test_binary, y_pred)
    cm = confusion_matrix(y_test_binary, y_pred)
    
    results_binary[name] = acc
    
    print(f"\nModel: {name}")
    print(f"CV Mean Accuracy: {cv_scores.mean():.4f}")
    print(f"Test Accuracy: {acc:.4f}")
    print("Confusion Matrix:")
    print(cm)

### 3.2 Multiclass Classification
**Target:** `diabetes_stage`
**Models:** Decision Tree, Random Forest, KNN

In [ ]:
# Target Preparation
y_train_multi = train_data['diabetes_stage']
y_test_multi = test_data['diabetes_stage']

# Encode target labels string -> int
le = LabelEncoder()
y_train_multi_enc = le.fit_transform(y_train_multi)
y_test_multi_enc = le.transform(y_test_multi)

# Models
multi_models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": KNeighborsClassifier()
}

results_multi = {}
print("--- Multiclass Classification Results ---")
for name, estimator in multi_models.items():
    model = make_model(estimator)
    # 5-fold CV
    cv_scores = cross_val_score(model, X_train_raw, y_train_multi_enc, cv=classification_cv, scoring='accuracy')
    
    # Fit and Predict
    model.fit(X_train_raw, y_train_multi_enc)
    y_pred = model.predict(X_test_raw)
    
    acc = accuracy_score(y_test_multi_enc, y_pred)
    results_multi[name] = acc
    
    print(f"\nModel: {name}")
    print(f"CV Mean Accuracy: {cv_scores.mean():.4f}")
    print(f"Test Accuracy: {acc:.4f}")
    # Confusion Matrix 
    print(confusion_matrix(y_test_multi_enc, y_pred))

### 3.3 Regression
**Target:** `diabetes_risk_score`
**Models:** Linear Regression, Ridge Regression, Random Forest Regressor

In [ ]:
# Target Preparation
y_train_reg = train_data['diabetes_risk_score']
y_test_reg = test_data['diabetes_risk_score']

# Models
reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest Reg": RandomForestRegressor(random_state=42)
}

results_regression = {}
print("--- Regression Results ---")
for name, estimator in reg_models.items():
    model = make_model(estimator)
    # 5-fold CV 
    cv_scores = cross_val_score(model, X_train_raw, y_train_reg, cv=regression_cv, scoring='neg_mean_squared_error')
    rmse_cv = np.sqrt(-cv_scores.mean())
    
    # Fit and Predict
    model.fit(X_train_raw, y_train_reg)
    y_pred = model.predict(X_test_raw)
    
    # Metrics
    mse = mean_squared_error(y_test_reg, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_reg, y_pred)
    results_regression[name] = {"rmse": rmse, "r2": r2}
    
    print(f"\nModel: {name}")
    print(f"CV Mean RMSE: {rmse_cv:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test R^2: {r2:.4f}")

## Historical study interpretation

**These are notes from the original study, not newly reproduced results. The original CSV files were not available during publication validation. Re-run the analysis on the original data before relying on the numerical statements below.**

## Module 4: Summary

### Findings

**1. Data Quality & Preprocessing**
The initial dataset was relatively clean, containing only a few missing values (approx. 8 rows) which were removed. Outlier detection using the IQR method resulted in the removal of 37 observations, leaving 4957 samples for training. Principal Component Analysis (PCA) was successfully applied to reduce dimensionality while retaining 95% of the variance, identifying 24 principal components from the original feature set.

**2. Visualization Insights**
The dataset shows a class imbalance, with a higher prevalence of diagnosed diabetes cases compared to non-diabetic cases. Strong correlations were observed between physical health indicators, specifically **BMI** and **Blood Pressure** metrics. Boxplot analysis confirmed that higher BMI levels are associated with advanced diabetes stages, particularly Type 2.

**3. Model Performance**

**Binary Classification (Predicting Diagnosed Diabetes):**
The models used were Logistic Regression, Random Forest, and SVM.
**Performance:** All three models performed similarly, achieving an accuracy between **61% and 62%**.
**Best Model:** Random Forest and Logistic Regression performed comparably (approx. 62.5% accuracy).
**Analysis:** The confusion matrices indicate significant misclassification, suggesting that the features (after PCA) might not be distinct enough to separate the two classes cleanly, or that the class imbalance is affecting the model's ability to generalize.

**Multiclass Classification (Predicting Diabetes Stage):**
The models used were Decision Tree, Random Forest, and KNN.
**Performance:** **Random Forest** achieved the highest accuracy at **60.4%**. KNN followed with ~53.8%, and Decision Tree performed the worst at ~49%.
**Analysis:** The confusion matrices show that the models struggle significantly with minority classes (likely Type 1 or Gestational diabetes), often predicting the majority class instead. This highlights the difficulty of distinguishing between specific diabetes subtypes using only general health indicators.

**Regression (Predicting Risk Score):**
The models used were Linear Regression, Ridge Regression, and Random Forest Regressor.
**Performance:** **Ridge Regression** and **Linear Regression** performed exceptionally well on the test set, achieving an **$R^2$ score of 0.91** and a low RMSE (~0.9). Random Forest performed slightly worse with an $R^2$ of 0.85.
**Analysis:** The high success of linear models suggests that the `diabetes_risk_score` is likely calculated using a linear combination of the provided features (e.g., Age, BMI, BP), making it easy for linear algorithms to predict.

### Conclusion

In this project, I successfully cleaned the Diabetes Health Indicators dataset and explored various machine learning approaches.

**Classification Challenge:** Predicting the exact diagnosis or stage of diabetes proved challenging with accuracies hovering around 60%. This suggests that while health indicators like BMI and BP are risk factors, they are not sole determinants; or, that class imbalance handling like SMOTE and hyperparameter tuning are needed to improve sensitivity.

**Regression Success:** The risk score prediction was highly accurate, indicating strong linear relationships between the input features and the calculated risk.

**Recommendation:** For future work, focusing on **Random Forest** for classification and **Ridge Regression** for risk scoring yields the best results. Additionally, addressing the class imbalance is crucial for improving the detection of specific diabetes stages.